<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_6_advanced_rag/week6_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Advanced Retrieval -- Transform, Re-rank, Measure

This extends the Week 5 Orbit retriever: same `small (120/20)` chunking, same 14-query set,
so the numbers below are directly comparable to what's already been measured.

**Transformation:** HyDE -- ask the model for a one-sentence hypothetical answer, then
retrieve using *that* answer's embedding instead of the raw query's. I used Claude instead (`claude-haiku-4-5-20251001`), the same swap already made and
disclosed for Week 5's generation step.

**Re-ranking:** a hybrid dense + lexical re-ranker. Over-retrieve a top-8 candidate pool
first, score each as `0.5 * embedding similarity + 0.5 * lexical overlap`, keep the top 3.
Over-retrieving matters here -- re-ranking a pool that's already just the top 3 can only
reorder it, never recover a chunk that got missed in the first place.

**Measuring it:** three variants per query -- baseline, transform-only, transform+rerank --
scored with the same precision@3/recall@3 method as Week 5. That splits the blame cleanly:
`delta_transform` is HyDE's effect, `delta_rerank` is the re-ranker's effect on top of that,
instead of one combined number that can't say which stage helped or hurt. Re-ranking is
scored against the real query, not the hypothetical -- the hypothetical is just a retrieval
proxy, not something the final ranking should actually match.


In [1]:
%pip install -q sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.6 MB/s eta 0:00:00


In [2]:
import os, re, pathlib, numpy as np
from collections import Counter
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

DOCS = {
    'networking': '''## Networking
The Orbit server listens on TCP port 7700 by default. Gossip runs on 7711.
Clients should connect to port 7700; port 7711 is for inter-node gossip only
and should not be exposed publicly.''',

    'storage': '''## Storage
Orbit persists data with an LSM-tree storage engine.
Writes go to an in-memory memtable first and are flushed to disk as
immutable SSTable segments once the memtable fills.''',

    'replication': '''## Replication
Each key is replicated with a default replication factor of 3.
Default consistency is quorum, meaning a write or read must be
acknowledged by at least two of the three replicas to succeed.''',

    'limits': '''## Limits
The maximum value size is 16 MB. A batch may contain at most 1000 operations.
Keys are limited to 1 KB. Requests exceeding any of these limits are
rejected with a client-side validation error before reaching storage.''',

    'security': '''## Security
Clients authenticate with API tokens. Traffic is encrypted with TLS.
Tokens are scoped per-namespace and can be revoked instantly through
the admin API without restarting the server.''',

    'backup': '''## Backup
Orbit takes incremental snapshots every 6 hours by default, written to
the configured backup directory. A full snapshot can be triggered manually
with orbitctl snapshot --full. Snapshots older than 7 days are pruned
automatically unless retention is overridden in orbit.yaml.''',

    'monitoring': '''## Monitoring
Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics.
Health checks are available at /healthz on the same port. The metrics
port is separate from the client port and is not encrypted by default.''',

    'deployment': '''## Deployment
Orbit is configured through a single YAML file, orbit.yaml, loaded at
startup. Environment variables prefixed with ORBIT_ override values from
the file. The default data directory is /var/lib/orbit, and the default
config search path is /etc/orbit/orbit.yaml.''',
}
print('doc count:', len(DOCS))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

doc count: 8


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out

chunks, sources = [], []
for name, text in DOCS.items():
    for c in chunk_fixed(text, 120, 20):
        chunks.append(c); sources.append(name)
print(f'small (120/20) -> {len(chunks)} chunks')

_vecs = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(_vecs.shape[1]); index.add(_vecs)

def embed_retrieve(q, k=3):
    """Plain dense retrieval -- the Week 5 baseline, unchanged."""
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    ids = index.search(qe, k)[1][0]
    return [int(i) for i in ids if i >= 0]

small (120/20) -> 22 chunks


In [4]:
import getpass

def _get_api_key():
    """Colab-safe key lookup: Colab secret, then env var, then an in-notebook prompt. Same
    pattern used for generation in Week 5 and the Week 5 RAGAS eval -- here it's needed for
    HyDE's generation step instead."""
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        return key
    try:
        return getpass.getpass('Enter your ANTHROPIC_API_KEY: ')
    except Exception:
        return None

_api_key = _get_api_key()
_client = None
if _api_key:
    import anthropic
    _client = anthropic.Anthropic(api_key=_api_key)
else:
    print('[no key found -- HyDE will fall back to the raw query, see hyde() below]')

def claude_chat(prompt, model='claude-haiku-4-5-20251001', max_tokens=200):
    if _client is None:
        return None
    resp = _client.messages.create(
        model=model, max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.content[0].text.strip()

## Part 1: Transform and re-rank

One transformation (HyDE) plus one re-rank step over the candidates

In [5]:
STOP = set('a an the of to in on for and or is are how do does what so that with by as it '
           'why can does should would be will'.split())

def hyde(q):
    """HyDE: ask Claude for a one-sentence hypothetical answer to the query, then return
    that text so the caller embeds and retrieves on IT instead of the raw query. The
    assumption HyDE makes is that a plausible-sounding generated answer is a good proxy for
    what a real, correct answer would look like -- which is exactly the assumption that
    breaks when the model answers generically or guesses wrong instead of admitting it
    doesn't know, since a generic or wrong hypothetical embeds nowhere near the chunk that
    actually has the fact. Falls back to the raw query if no API key was available."""
    if _client is None:
        return q
    answer = claude_chat(f'Write a single, direct, one-sentence hypothetical answer to this '
                         f'question about the Orbit server. Answer as if you know the fact, '
                         f'even if you are guessing -- do not hedge or say you are unsure.\n\n'
                         f'Question: {q}')
    return answer if answer else q

def lexical_score(q, chunk):
    """Fraction of the query's non-stopword terms that appear verbatim in the chunk --
    a simple lexical-overlap signal, independent of the embedding space."""
    q_terms = set(re.findall(r'[a-zA-Z0-9]+', q.lower())) - STOP
    c_terms = set(re.findall(r'[a-zA-Z0-9]+', chunk.lower()))
    if not q_terms:
        return 0.0
    return len(q_terms & c_terms) / len(q_terms)

def rerank(q, candidate_ids, k=3, alpha=0.5):
    """Re-score an over-retrieved candidate pool with a dense+lexical blend, then keep the
    new top-k. Reranking a pool larger than k (not just the original top-k) is what lets this
    step actually recover a relevant chunk the first-pass embedding search ranked below k."""
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')[0]
    ce = embedder.encode([chunks[i] for i in candidate_ids], normalize_embeddings=True)
    scored = []
    for idx, i in enumerate(candidate_ids):
        dense = float(ce[idx] @ qe)
        lex = lexical_score(q, chunks[i])
        scored.append((alpha * dense + (1 - alpha) * lex, i))
    scored.sort(key=lambda t: -t[0])
    return [i for _, i in scored[:k]]

def run_baseline(q, k=3):
    return embed_retrieve(q, k=k)

def run_transform_only(q, k=3):
    return embed_retrieve(hyde(q), k=k)

def run_transform_rerank(q, k=3, over_retrieve=8):
    hypothetical = hyde(q)
    candidates = embed_retrieve(hypothetical, k=over_retrieve)
    return rerank(q, candidates, k=k)

## Part 2: Measure the effect

Same 14 queries as Week 5 Part 2, same ground truth (a chunk counts as relevant if it
contains the query's answer fact). Three variants per query -- baseline, transform-only,
transform+rerank -- so we can tell the transformation's effect apart from the re-ranker's
instead of just seeing one combined number.

In [6]:
QA = [
    ('what port does orbit listen on for clients', '7700', 'networking'),
    ('what port is used for gossip between nodes', '7711', 'networking'),
    ('what storage engine does orbit use', 'LSM-tree', 'storage'),
    ('what happens when the memtable fills up', 'flushed to disk', 'storage'),
    ('what is the default replication factor', 'replication factor of 3', 'replication'),
    ('what consistency level is used by default', 'quorum', 'replication'),
    ('what is the maximum value size', '16 MB', 'limits'),
    ('how many operations can a batch contain', '1000 operations', 'limits'),
    ('how do clients authenticate', 'API tokens', 'security'),
    ('how often are snapshots taken', 'every 6 hours', 'backup'),
    ('what port serves metrics', '7720', 'monitoring'),
    ('where can i check server health', '/healthz', 'monitoring'),
    ('how is orbit configured', 'configured through a single YAML file', 'deployment'),
    ('what environment variable prefix overrides config', 'ORBIT_', 'deployment'),
]
print(f'{len(QA)} queries (same set as Week 5 Part 2)')

def chunk_relevance(fact):
    return {i for i, c in enumerate(chunks) if fact.lower() in c.lower()}

def precision_recall(retrieved_ids, fact):
    retrieved = set(retrieved_ids)
    relevant = chunk_relevance(fact)
    tp = len(retrieved & relevant)
    precision = tp / len(retrieved) if retrieved else 0.0
    recall = tp / len(relevant) if relevant else 0.0
    return precision, recall

rows = []
for q, fact, doc in QA:
    pb, rb = precision_recall(run_baseline(q), fact)
    pt, rt = precision_recall(run_transform_only(q), fact)
    pf, rf = precision_recall(run_transform_rerank(q), fact)
    rows.append({
        'query': q,
        'baseline_p': pb, 'baseline_r': rb,
        'transform_p': pt, 'transform_r': rt,
        'final_p': pf, 'final_r': rf,
        'delta_transform': pt - pb,
        'delta_rerank': pf - pt,
        'delta_total': pf - pb,
    })
    print(f'{q!r}')
    print(f'  precision@3: baseline={pb:.2f}  transform={pt:.2f}  final={pf:.2f}   '
          f'(d_transform={pt - pb:+.2f}  d_rerank={pf - pt:+.2f}  d_total={pf - pb:+.2f})')
    print(f'  recall@3:    baseline={rb:.2f}  transform={rt:.2f}  final={rf:.2f}')
    print()

print('=== averages ===')
for key, label in [('baseline_p', 'precision@3 baseline'), ('transform_p', 'precision@3 transform-only'),
                    ('final_p', 'precision@3 final'), ('baseline_r', 'recall@3 baseline'),
                    ('transform_r', 'recall@3 transform-only'), ('final_r', 'recall@3 final')]:
    avg = sum(r[key] for r in rows) / len(rows)
    print(f'{label:28s} {avg:.3f}')

improved = sum(1 for r in rows if r['delta_transform'] > 0)
regressed = sum(1 for r in rows if r['delta_transform'] < 0)
unchanged = sum(1 for r in rows if r['delta_transform'] == 0)
print(f'\ntransformation alone: {improved} improved, {regressed} regressed, {unchanged} unchanged '
      f'(a regression here is the required Part 3 failure case)')

14 queries (same set as Week 5 Part 2)
'what port does orbit listen on for clients'
  precision@3: baseline=0.33  transform=0.33  final=0.33   (d_transform=+0.00  d_rerank=+0.00  d_total=+0.00)
  recall@3:    baseline=0.50  transform=0.50  final=0.50

'what port is used for gossip between nodes'
  precision@3: baseline=0.67  transform=0.67  final=0.67   (d_transform=+0.00  d_rerank=+0.00  d_total=+0.00)
  recall@3:    baseline=1.00  transform=1.00  final=1.00

'what storage engine does orbit use'
  precision@3: baseline=0.33  transform=0.33  final=0.33   (d_transform=+0.00  d_rerank=+0.00  d_total=+0.00)
  recall@3:    baseline=1.00  transform=1.00  final=1.00

'what happens when the memtable fills up'
  precision@3: baseline=0.33  transform=0.33  final=0.33   (d_transform=+0.00  d_rerank=+0.00  d_total=+0.00)
  recall@3:    baseline=1.00  transform=1.00  final=1.00

'what is the default replication factor'
  precision@3: baseline=0.33  transform=0.33  final=0.33   (d_transform=+0.00  

## Results

Real run, 14 queries, Claude as the HyDE generator:

| | precision@3 | recall@3 |
|---|---|---|
| baseline | 0.381 | 0.964 |
| transform-only (HyDE) | 0.286 | 0.750 |
| final (HyDE + rerank) | 0.310 | 0.821 |

HyDE never helped here -- 0 improved, 3 regressed, 11 unchanged. Re-ranking clawed back some
of the damage (precision 0.286 -> 0.310, recall 0.750 -> 0.821) but not all of it; both final
numbers are still below baseline.

Kind of the opposite of how HyDE is usually pitched: baseline was already close to ceiling
(recall@3 0.964), so there wasn't much room to gain and plenty of room to lose. All three
regressions hit queries with a short, specific answer that's already close to the query's own
wording -- "quorum," "API tokens," "/healthz." Paraphrasing a query that's already easy to
match just adds a chance to drift away from it.

Re-ranking only fully fixed one of the three: "how do clients authenticate" went from a full
miss back to a perfect hit (recall 1.00 -> 0.00 -> 1.00). The other two stayed broken -- but,
as Part 3 shows, for two different reasons, not the same one.

## Part 3: Find one failure and explain it

**Failure case: "where can i check server health" (`/healthz`).** Baseline got it right
(precision 0.33, recall 1.00). HyDE alone dropped it to 0/0, and re-ranking didn't bring it
back.

Here's what Claude actually generated for the three regressed queries:

> `where can i check server health` -> "You can check Orbit server health by accessing the
> monitoring dashboard at `https://your-orbit-instance/admin/health` or through your server's
> status page in the administrative console."

> `what consistency level is used by default` -> "The Orbit server uses strong consistency as
> its default consistency level."

> `how do clients authenticate` -> "Clients authenticate to the Orbit server by providing
> credentials (typically an API key or token) in the request headers or as a query parameter."

That first one is the clearest failure: Claude didn't hedge, it invented a specific,
convincing-sounding URL and admin console that don't exist anywhere in the docs. The real
answer is just `/healthz`. That's the core risk with HyDE -- a generated guess is only a good
stand-in for a real answer when it's actually right, and a confident wrong guess is worse
than a vague one, since it reads like real documentation and embeds nowhere near the chunk
that actually has the fact.

I checked this directly instead of assuming it: retrieving on the `/healthz` hypothetical
never pulls in the chunk that actually has `/healthz`, even with 8 candidates instead of 3 --
so re-ranking never got a shot at it.

"What consistency level is used by default" looked identical on paper (same 1.00 -> 0.00 ->
0.00 pattern, same kind of confident-but-wrong guess -- "strong consistency" instead of
"quorum") but the same check tells a different story: the chunk with "quorum" *is* in that
candidate pool. So this one wasn't a retrieval miss -- it's a re-ranking miss. The real
query's dense+lexical score just ranked something else above it.

"How do clients authenticate" regressed the same way but recovered fully after re-ranking,
and the quote explains why: buried in Claude's invented elaboration, it still says "API key
or token," close enough to the doc's "API tokens" that the right chunk survived into the
pool -- same as the quorum case -- except this time the real query's overlap with it was
strong enough to win the re-rank.

So two genuinely different failures, not one: one query lost its answer at the HyDE step,
before re-ranking ever got a turn; the other lost it at the re-ranking step, despite the right
answer being available. Fixing the first means not trusting HyDE's retrieval alone -- e.g.
merging in the raw query's own candidates too, so a bad guess can't unilaterally exclude a
chunk the raw query would've found. Fixing the second is a re-ranker tuning problem, not a
transformation one.

In [7]:
for q in ['where can i check server health',
          'what consistency level is used by default',
          'how do clients authenticate']:
    print(f'{q!r} -> {hyde(q)!r}')

'where can i check server health' -> "You can check Orbit server health by accessing the monitoring dashboard at `https://your-orbit-instance/admin/health` or through your server's status page in the administrative console."
'what consistency level is used by default' -> 'The Orbit server uses strong consistency as its default consistency level.'
'how do clients authenticate' -> 'Clients authenticate to the Orbit server by providing credentials (typically an API key or token) in the request headers or as a query parameter.'


In [8]:
cases = {
    'where can i check server health': (
        "You can check Orbit server health by accessing the monitoring dashboard at "
        "`https://your-orbit-instance/admin/health` or through your server's status page "
        "in the administrative console.",
        '/healthz',
    ),
    'what consistency level is used by default': (
        'The Orbit server uses strong consistency as its default consistency level.',
        'quorum',
    ),
}

for q, (hypothetical, fact) in cases.items():
    pool = embed_retrieve(hypothetical, k=8)
    relevant = chunk_relevance(fact)
    hit = bool(set(pool) & relevant)
    print(f'{q!r}')
    print(f'  candidate pool (k=8) under the hypothetical: {pool}')
    print(f'  relevant chunk id(s) for {fact!r}: {sorted(relevant)}')
    print(f'  relevant chunk present in pool: {hit}')
    print()

'where can i check server health'
  candidate pool (k=8) under the hypothetical: [16, 20, 21, 0, 13, 14, 19, 3]
  relevant chunk id(s) for '/healthz': [17]
  relevant chunk present in pool: False

'what consistency level is used by default'
  candidate pool (k=8) under the hypothetical: [0, 14, 13, 21, 3, 20, 5, 16]
  relevant chunk id(s) for 'quorum': [5, 6]
  relevant chunk present in pool: True



## Part 4: schema-constrained extraction

This adds a separate, typed-extraction path that can answer two structured queries vector
search alone struggles with: a **count** ("how many distinct ports does Orbit define?") and
a **cross-doc lookup** ("is the max batch size bigger than the replication factor?"). Both
need facts pulled from more than one doc at once, which top-k retrieval isn't built for -- it
just returns the chunks most similar to the query, not every chunk the answer actually needs.

**Schema:** three entity types -- `port`, `limit`, `setting` -- each with a name, a literal
value, and an optional unit. Small on purpose, just enough to answer the two queries below.

**"Schema-constrained" means:** the entities come from a Claude tool-use call with
`tool_choice` forcing the schema, so the model's output has to match it at generation time
instead of being parsed out of free text afterward -- same mechanism as the guarded tools in
Week 4.

In [9]:
ENTITY_TOOL = {
    'name': 'extract_entities',
    'description': ('Extract structured entities (ports, limits, and settings) that are '
                     'explicitly and literally stated in the given Orbit documentation '
                     'section. Only include values literally present in the text -- do not '
                     'infer, guess, or compute a value that is not directly stated.'),
    'input_schema': {
        'type': 'object',
        'properties': {
            'entities': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'entity_type': {'type': 'string', 'enum': ['port', 'limit', 'setting']},
                        'name': {'type': 'string',
                                  'description': "short label, e.g. 'client port' or 'maximum batch size'"},
                        'value': {'type': 'string',
                                   'description': "the literal value as stated, e.g. '7700' or 'quorum'"},
                        'unit': {'type': 'string',
                                  'description': "unit if any, e.g. 'MB', 'operations', 'days'; empty string if none"},
                    },
                    'required': ['entity_type', 'name', 'value', 'unit'],
                },
            },
        },
        'required': ['entities'],
    },
}

def extract_entities_from_doc(doc_name):
    """One schema-constrained extraction call per doc. Returns [] gracefully if no API key
    was available (same degrade-gracefully pattern as hyde() above)."""
    if _client is None:
        return []
    resp = _client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=500,
        tools=[ENTITY_TOOL], tool_choice={'type': 'tool', 'name': 'extract_entities'},
        messages=[{'role': 'user',
                    'content': f'Extract entities from this Orbit documentation section:\n\n{DOCS[doc_name]}'}],
    )
    for block in resp.content:
        if block.type == 'tool_use':
            return [dict(e, source_doc=doc_name) for e in block.input.get('entities', [])]
    return []

ENTITIES = []
for _doc_name in DOCS:
    ENTITIES.extend(extract_entities_from_doc(_doc_name))

print(f'extracted {len(ENTITIES)} entities across {len(DOCS)} docs:')
for e in ENTITIES:
    print(' ', e)

extracted 20 entities across 8 docs:
  {'entity_type': 'port', 'name': 'Orbit server TCP port', 'value': '7700', 'unit': '', 'source_doc': 'networking'}
  {'entity_type': 'port', 'name': 'Gossip port', 'value': '7711', 'unit': '', 'source_doc': 'networking'}
  {'entity_type': 'port', 'name': 'Client connection port', 'value': '7700', 'unit': '', 'source_doc': 'networking'}
  {'entity_type': 'setting', 'name': 'replication factor', 'value': '3', 'unit': '', 'source_doc': 'replication'}
  {'entity_type': 'setting', 'name': 'default consistency', 'value': 'quorum', 'unit': '', 'source_doc': 'replication'}
  {'entity_type': 'limit', 'name': 'minimum replicas for write acknowledgment', 'value': '2', 'unit': 'replicas', 'source_doc': 'replication'}
  {'entity_type': 'limit', 'name': 'minimum replicas for read acknowledgment', 'value': '2', 'unit': 'replicas', 'source_doc': 'replication'}
  {'entity_type': 'limit', 'name': 'maximum value size', 'value': '16', 'unit': 'MB', 'source_doc': 'limi

In [10]:
def answer_port_count(table):
    ports = {e['value'] for e in table if e['entity_type'] == 'port'}
    docs = sorted({e['source_doc'] for e in table if e['entity_type'] == 'port'})
    return len(ports), sorted(ports), docs

def answer_batch_vs_replication(table):
    batch = next((e for e in table if 'batch' in e['name'].lower()), None)
    repl = next((e for e in table if 'replication factor' in e['name'].lower()), None)
    if batch is None or repl is None:
        return None
    return int(batch['value']) > int(repl['value']), batch['value'], repl['value']

n_ports, port_values, port_docs = answer_port_count(ENTITIES)
print(f'Q1 (count): Orbit defines {n_ports} distinct ports: {port_values}, sourced from: {port_docs}')

_cmp = answer_batch_vs_replication(ENTITIES)
if _cmp:
    is_greater, batch_val, repl_val = _cmp
    verb = 'is' if is_greater else 'is NOT'
    print(f'Q2 (cross-doc lookup): max batch size ({batch_val}) {verb} greater than '
          f'the default replication factor ({repl_val})')
else:
    print('Q2: could not find both the batch-size and replication-factor entities '
          '-- extraction may have used different wording than expected, inspect ENTITIES above')

Q1 (count): Orbit defines 3 distinct ports: ['7700', '7711', '7720'], sourced from: ['monitoring', 'networking']
Q2 (cross-doc lookup): max batch size (1000) is greater than the default replication factor (3)


In [11]:
def baseline_doc_coverage(q, k=3):
    ids = embed_retrieve(q, k=k)
    return sorted({sources[i] for i in ids})

port_q = 'how many distinct ports does orbit define across all of its documentation'
batch_q = 'is the maximum batch size greater than the default replication factor'

print('baseline vector retrieval doc coverage (top-3), real embedder:')
print(f'  port-count query:    {baseline_doc_coverage(port_q)}  (needs networking AND monitoring)')
print(f'  batch-vs-repl query: {baseline_doc_coverage(batch_q)}  (needs limits AND replication)')

baseline vector retrieval doc coverage (top-3), real embedder:
  port-count query:    ['deployment', 'networking']  (needs networking AND monitoring)
  batch-vs-repl query: ['limits', 'replication']  (needs limits AND replication)


## Results

Real extraction pulled **20** entities (health/metrics
endpoints, config paths), plus some duplication: the same port (7700) got pulled out twice
because it's stated twice in the doc, and the quorum sentence became two separate "minimum
replicas" entities.

None of that broke anything, because the query functions dedupe and match loosely on purpose:

```
Q1 (count): Orbit defines 3 distinct ports: ['7700', '7711', '7720'], sourced from: ['monitoring', 'networking']
Q2 (cross-doc lookup): max batch size (1000) is greater than the default replication factor (3)
```

Both right. The baseline-retrieval check, though, only backs up the claim for one of the two
queries. For the port count, vector search actually did miss a needed doc -- its top 3 had
`deployment` and `networking` but not `monitoring`, so a plain RAG answer would likely come
back "2 ports," wrong without anyone noticing. For the batch-vs-replication comparison,
baseline retrieval actually got both docs it needed (`limits` and `replication`) -- so that
one didn't demonstrably fail this time.

Worth saying plainly instead of forcing both into the same story: the extraction approach is
still correct, and correct *by construction* regardless of retrieval luck -- but only the
count query actually proves vector search needed fixing here. The other one just shows the
extraction approach works, not that it was necessary.